# Correction — Régression et baseline

La comparaison garde le même split et les mêmes métriques pour une prédiction constante et une droite. On visualise les deux fonctions apprises pour comprendre ce que gagne un modèle linéaire.

## Formules et protocole

La MSE est la moyenne de `(target-prediction)**2` ; elle est exprimée dans l'unité de la cible au carré. La MAE conserve l'unité de la cible. Les gradients calculés dans la première cellule minimisent la MSE sur le jeu synthétique. Pour comparer les modèles, les deux estimateurs de bibliothèque sont ajustés sur `x_train` seulement.

In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(17)
x = np.linspace(0, 5, 120)
y = 2.4 * x + 1 + rng.normal(0, 0.5, len(x))
x_train, x_test, y_train, y_test = train_test_split(x[:, None], y, random_state=9)

def mse(target, prediction):
    return np.mean((target - prediction) ** 2)

# Même jeu train pour la descente maison et l'implémentation de bibliothèque.
w, b = 0.0, 0.0
train_feature = x_train[:, 0]
for _ in range(300):
    error = w * train_feature + b - y_train
    w -= 0.01 * 2 * np.mean(error * train_feature)
    b -= 0.01 * 2 * np.mean(error)
reference = LinearRegression().fit(x_train, y_train)
print('scratch:', round(w, 3), round(b, 3))
print('library:', round(reference.coef_[0], 3), round(reference.intercept_, 3))

for estimator in [DummyRegressor(strategy='mean'), LinearRegression()]:
    estimator.fit(x_train, y_train)
    pred = estimator.predict(x_test)
    print(type(estimator).__name__, 'MAE:', round(mean_absolute_error(y_test, pred), 3), 'MSE:', round(mse(y_test, pred), 3))

## Deux prédictions sur le même test

La baseline donne une ligne horizontale : elle ignore `x`. La régression peut suivre une tendance. Regardez aussi l'écart vertical entre points et lignes, qui correspond aux résidus.

In [ ]:
import matplotlib.pyplot as plt
baseline_model = DummyRegressor(strategy='mean').fit(x_train, y_train)
linear_model = LinearRegression().fit(x_train, y_train)
order = np.argsort(x_test[:, 0])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_test[:, 0], y_test, s=20, color='black', alpha=0.7, label='Test')
ax.plot(x_test[order, 0], baseline_model.predict(x_test[order]), label='Baseline : moyenne train')
ax.plot(x_test[order, 0], linear_model.predict(x_test[order]), label='Régression linéaire')
ax.set(xlabel='x (unité arbitraire)', ylabel='y (unité arbitraire)', title='Deux modèles, même partition test')
ax.legend(); ax.grid(alpha=0.2)
plt.show()

## Interprétation et limite

Si la droite réduit MAE/MSE sur ce split, elle capte un signal que la moyenne ignore. Le jeu est construit à partir d'une droite : cette réussite ne dit pas que toute relation réelle sera linéaire. Les résidus et un autre split restent utiles pour juger la stabilité.